# Fruit-360 Classification with MobileNetV2

Notebook Google Colab hoàn chỉnh để phân loại ảnh trái cây bằng **MobileNetV2 Transfer Learning** trên dataset **Fruit-360**.

Luồng xử lý:
1. Mount Google Drive
2. Tự tìm hoặc cấu hình file zip dataset Fruit-360
3. Unzip dataset
4. Load dữ liệu Training / Test
5. Train MobileNetV2
6. Evaluate
7. Vẽ confusion matrix
8. Lưu model và class labels vào Google Drive

## 1. Kiểm tra GPU

In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU devices:", gpus)

if not gpus:
    print("WARNING: Chưa bật GPU. Vào Runtime > Change runtime type > Hardware accelerator > T4 GPU.")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Cấu hình đường dẫn dataset và output

Mặc định notebook sẽ tìm file `.zip` có tên chứa `fruit` trong `/content/drive/MyDrive`. Nếu muốn chỉ định chính xác, sửa biến `DRIVE_ZIP_PATH`.

In [ ]:
from pathlib import Path
import os
import json
import zipfile

# Nếu biết chính xác đường dẫn zip, sửa tại đây. Ví dụ:
# DRIVE_ZIP_PATH = "/content/drive/MyDrive/fruits-360_dataset_100x100.zip"
DRIVE_ZIP_PATH = ""  # để trống thì notebook tự tìm file zip có tên chứa 'fruit'

DRIVE_ROOT = Path("/content/drive/MyDrive")
WORK_DIR = Path("/content/fruit360_project")
DATA_DIR = WORK_DIR / "dataset"
OUTPUT_DIR = DRIVE_ROOT / "Fruit360_MobileNetV2_Output"

WORK_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def find_fruit_zip(root: Path) -> Path:
    candidates = sorted(root.rglob("*.zip"))
    fruit_zips = [p for p in candidates if "fruit" in p.name.lower() or "fruits" in p.name.lower()]
    if not fruit_zips:
        raise FileNotFoundError(
            "Không tìm thấy file zip Fruit-360 trong Google Drive. "
            "Hãy upload dataset zip vào MyDrive hoặc sửa biến DRIVE_ZIP_PATH."
        )
    return fruit_zips[0]

zip_path = Path(DRIVE_ZIP_PATH) if DRIVE_ZIP_PATH else find_fruit_zip(DRIVE_ROOT)
print("Dataset zip:", zip_path)
print("Extract dir:", DATA_DIR)
print("Output dir:", OUTPUT_DIR)

## 4. Tự unzip dataset Fruit-360

In [ ]:
def has_extracted_data(root: Path) -> bool:
    return any(p.is_dir() and p.name.lower() == "training" for p in root.rglob("*")) and \
           any(p.is_dir() and p.name.lower() in {"test", "testing"} for p in root.rglob("*"))

if has_extracted_data(DATA_DIR):
    print("Dataset đã được giải nén trước đó, bỏ qua unzip.")
else:
    print("Đang giải nén dataset...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(DATA_DIR)
    print("Giải nén xong.")

print("Một số thư mục con:")
for p in list(DATA_DIR.rglob("*"))[:20]:
    if p.is_dir():
        print("-", p)

## 5. Tự xác định thư mục Training và Test

In [ ]:
def find_dataset_split(root: Path, names) -> Path:
    names = {name.lower() for name in names}
    matches = [p for p in root.rglob("*") if p.is_dir() and p.name.lower() in names]
    if not matches:
        raise FileNotFoundError(f"Không tìm thấy thư mục {names} trong {root}")
    # Chọn thư mục có nhiều class nhất.
    return max(matches, key=lambda p: sum(1 for child in p.iterdir() if child.is_dir()))

TRAIN_DIR = find_dataset_split(DATA_DIR, ["Training", "train"])
TEST_DIR = find_dataset_split(DATA_DIR, ["Test", "Testing", "test"])

print("TRAIN_DIR:", TRAIN_DIR)
print("TEST_DIR:", TEST_DIR)
print("Số class train:", len([p for p in TRAIN_DIR.iterdir() if p.is_dir()]))
print("Số class test:", len([p for p in TEST_DIR.iterdir() if p.is_dir()]))

## 6. Load dataset bằng TensorFlow

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

IMG_SIZE = (160, 160)
BATCH_SIZE = 32
SEED = 42
VALIDATION_SPLIT = 0.2

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=VALIDATION_SPLIT,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=VALIDATION_SPLIT,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical"
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    label_mode="categorical"
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)

print("NUM_CLASSES:", NUM_CLASSES)
print("First 10 classes:", class_names[:10])

with open(OUTPUT_DIR / "class_names.json", "w", encoding="utf-8") as f:
    json.dump(class_names, f, ensure_ascii=False, indent=2)

## 7. Tối ưu pipeline dữ liệu và xem mẫu ảnh

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

plt.figure(figsize=(12, 8))
for images, labels in train_ds.take(1):
    for i in range(12):
        ax = plt.subplot(3, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        label_idx = int(np.argmax(labels[i].numpy()))
        plt.title(class_names[label_idx], fontsize=8)
        plt.axis("off")
plt.tight_layout()
plt.show()

## 8. Xây dựng model MobileNetV2

In [ ]:
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.08),
], name="data_augmentation")

base_model = MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.25)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs, name="fruit360_mobilenetv2")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## 9. Train phần classification head

In [ ]:
INITIAL_EPOCHS = 8

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(OUTPUT_DIR / "best_fruit360_mobilenetv2.keras"),
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=INITIAL_EPOCHS,
    callbacks=callbacks
)

## 10. Fine-tune MobileNetV2

In [ ]:
base_model.trainable = True

# Chỉ fine-tune các layer cuối để tránh overfit và giảm thời gian train.
FINE_TUNE_AT = 100
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

FINE_TUNE_EPOCHS = 8
TOTAL_EPOCHS = INITIAL_EPOCHS + FINE_TUNE_EPOCHS

fine_tune_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=TOTAL_EPOCHS,
    initial_epoch=len(history.epoch),
    callbacks=callbacks
)

## 11. Vẽ biểu đồ quá trình train

In [ ]:
def merge_histories(*histories):
    result = {}
    for h in histories:
        for key, values in h.history.items():
            result.setdefault(key, []).extend(values)
    return result

hist = merge_histories(history, fine_tune_history)

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(hist["accuracy"], label="train_acc")
plt.plot(hist["val_accuracy"], label="val_acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(hist["loss"], label="train_loss")
plt.plot(hist["val_loss"], label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_history.png", dpi=160)
plt.show()

## 12. Evaluate trên test set

In [ ]:
best_model_path = OUTPUT_DIR / "best_fruit360_mobilenetv2.keras"
if best_model_path.exists():
    model = tf.keras.models.load_model(best_model_path)
    print("Loaded best model:", best_model_path)

test_loss, test_acc = model.evaluate(test_ds, verbose=1)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

metrics = {
    "test_loss": float(test_loss),
    "test_accuracy": float(test_acc),
    "num_classes": int(NUM_CLASSES),
    "image_size": list(IMG_SIZE),
    "batch_size": int(BATCH_SIZE)
}
with open(OUTPUT_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

## 13. Classification report và confusion matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

y_true = []
y_pred = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

report = classification_report(y_true, y_pred, target_names=class_names, digits=4)
print(report)

with open(OUTPUT_DIR / "classification_report.txt", "w", encoding="utf-8") as f:
    f.write(report)

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(max(12, NUM_CLASSES * 0.18), max(10, NUM_CLASSES * 0.18)))
sns.heatmap(
    cm,
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    cbar=True,
    square=True
)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Fruit-360 Confusion Matrix")
plt.xticks(rotation=90, fontsize=6)
plt.yticks(rotation=0, fontsize=6)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=220)
plt.show()

## 14. Lưu model hoàn chỉnh

In [ ]:
final_keras_path = OUTPUT_DIR / "fruit360_mobilenetv2_final.keras"
model.save(final_keras_path)
print("Saved Keras model:", final_keras_path)

# Export SavedModel để phục vụ deployment TensorFlow Serving / chuyển đổi nâng cao.
saved_model_dir = OUTPUT_DIR / "fruit360_mobilenetv2_savedmodel"
model.export(saved_model_dir)
print("Exported SavedModel:", saved_model_dir)

## 15. Chuyển sang TensorFlow Lite

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

tflite_path = OUTPUT_DIR / "fruit360_mobilenetv2.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

print("Saved TFLite model:", tflite_path)

## 16. Test dự đoán một ảnh bất kỳ

In [ ]:
from tensorflow.keras.utils import load_img, img_to_array
import random

all_test_images = []
for ext in ("*.jpg", "*.jpeg", "*.png"):
    all_test_images.extend(TEST_DIR.rglob(ext))

sample_path = random.choice(all_test_images)
img = load_img(sample_path, target_size=IMG_SIZE)
arr = img_to_array(img)
arr_batch = np.expand_dims(arr, axis=0)

pred = model.predict(arr_batch, verbose=0)[0]
top_idx = int(np.argmax(pred))
confidence = float(pred[top_idx])

plt.figure(figsize=(4, 4))
plt.imshow(img)
plt.axis("off")
plt.title(f"Pred: {class_names[top_idx]} ({confidence:.2%})")
plt.show()

print("Image:", sample_path)
print("Predicted class:", class_names[top_idx])
print("Confidence:", confidence)

## 17. Tổng kết file output

Sau khi chạy xong, các file sẽ nằm trong thư mục Google Drive:

`/content/drive/MyDrive/Fruit360_MobileNetV2_Output`

Bao gồm:
- `best_fruit360_mobilenetv2.keras`
- `fruit360_mobilenetv2_final.keras`
- `fruit360_mobilenetv2_savedmodel/`
- `fruit360_mobilenetv2.tflite`
- `class_names.json`
- `metrics.json`
- `classification_report.txt`
- `training_history.png`
- `confusion_matrix.png`